In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [23]:
fbref_url = "https://fbref.com{}"
premier_league = fbref_url.format("/en/comps/9/Premier-League-Stats")
data = requests.get(premier_league)

## Retrieving Individual Player Data

In [796]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import pickle
from IPython.display import clear_output


def make_request(endpoint: str) -> str:
    while True:
        response = requests.get(endpoint)
        if response.status_code == 429:  # Rate limited
            print("Rate limited. Pausing for 5 minutes...")
            print(endpoint)
            print(response.text)
            time.sleep(300)  # Wait for 5 minutes
        else:
            return response.text

def get_page_soup(url:str) -> BeautifulSoup:
    return BeautifulSoup(make_request(url), 'html.parser')

def extract_team_links(url:str) -> dict:
    league_teams_soup = get_page_soup(url)
    league_table = league_teams_soup.select('table.stats_table')[0]
    teams_dict = {team_link.get_text():fbref_url.format(team_link.get("href")) for team_link in league_table.find_all('a') if "squads" in team_link.get("href")}
    return teams_dict

def extract_player_links(url:str) -> dict:
    team_soup = get_page_soup(url)
    team_players_table = team_soup.select('table.stats_table')[0]
    players_links_dict = {
            l.get_text():fbref_url.format(l.get("href")) for l in team_players_table.find_all('a') 
            if (link := l.get("href")) and "players" in link and "matchlogs" not in link
        }
    return players_links_dict

def extract_player_details(url:str) -> dict:
    player_soup = get_page_soup(url)
    media_item_div = player_soup.select_one('div.media-item')
    image_src = media_item_div.find('img')['src'] if media_item_div else None

    # Extracting data dynamically
    for p in player_soup.find_all('p'):
        text = p.get_text(strip=True)
        # Extract position and footed
        if "Position:" in text:
            position_part = text.split("▪")[0].strip()
            footed_part = text.split("▪")[1].strip() if "▪" in text else None
            player_data['position'] = position_part.split(":")[1].strip()
            if footed_part:
                player_data['footed'] = footed_part.split(":")[1].strip()
        # Extract height and weight
        if "cm" in text and "kg" in text:
            height, weight = map(str.strip, text.split(',')[:2])
            player_data['height'] = height
            player_data['weight'] = weight.split('(')[0]
        # Extract date of birth and place of birth
        if "Born:" in text:
            birth_data = p.find("span", {"data-birth": True})
            if birth_data:
                player_data['dob'] = birth_data["data-birth"]
                birth_place = birth_data.find_next_sibling("span")
                if birth_place:
                    player_data['birth_place'] = birth_place.get_text(strip=True)[3:]  # Clean prefix if necessary
        # Extract national team
        if "National Team:" in text:
            player_data['national_team'] = text.split(":")[1].strip()
        # Extract club
        if "Club:" in text:
            player_data['club'] = text.split(":")[1].strip()
        # Extract wages and contract expiry
        if "Wages" in text:
            wages = p.select_one('.important').get_text(strip=True) if p.select_one('.important') else None
            player_data['wages'] = wages
            if "Expires" in text:
                contract_expires = text.split("Expires")[1].split(".")[0].strip()
                player_data['contract_expiry'] = contract_expires
    if image_src:
        player_data['img'] = image_src
    return player_data

# URLs
premier_league_links = [
    'https://fbref.com/en/comps/9/Premier-League-Stats',
    'https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2021-2022/2021-t2022-Premier-League-Stats'
]

fbref_url = 'https://fbref.com{}'

# Load existing player list from CSV

all_players_dict = pd.read_csv('players.csv', index_col=0).to_dict(orient='index')

for premier_league_link in premier_league_links:
    # Fetch Premier League standings
    premier_league_year = premier_league_link.split('/')[-2]
    team_links = extract_team_links(premier_league_link)
    team_dict = {}
    for team_name, team_link in team_links.items():
        player_links = extract_player_links(team_link)
        team_players_dict = {}
        for player_name, player_link in player_links.items():
            if player_name in all_players_dict:
                print(f"{player_name} already in list. Skipping...")
                team_players_dict[player_name] = all_players_dict[player_name]
            else:
                player_data = extract_player_details(player_link)    
                team_players_dict[player_name] = player_data
                print(f"Retrieved player data from fbref {player_name}: {player_data}")
                all_players_dict[player_name] = player_data
                time.sleep(4)  
            
        clear_output(wait=True)
        team_dict[team_name] = team_players_dict
        print(squad_players_dict)
        time.sleep(5)
        clear_output(wait=True)
    pd.DataFrame.from_dict(all_players_dict).to_csv(f'players{premier_league_year }.csv', index=False)
    with open(f'team_dict_{premier_league_year}.pkl', 'wb') as f:
        pickle.dump(team_dict, f)

    print(f"Finished processing season: {premier_league_link.split('/')[-2]}")
    time.sleep(60)


Finished processing season: 9


KeyboardInterrupt: 

## Get All Matches

In [885]:
import io

def get_fixtures(url:str) -> BeautifulSoup:
    fixtures_soup = get_page_soup(url)
    fixtures_table = fixtures_soup.find(id='all_sched')
    return fixtures_table

def get_match_data(url:str) -> list:
    game_soup = get_page_soup(url)
    team_stats = list()
    team_game_stats = game_soup.find_all(id=re.compile("all_player_stats"))
    goaly_game_stats = game_soup.find_all(id=re.compile("keeper_stats"), class_ = "table_wrapper")
    for team_stat in team_game_stats:
        team_stats.append(pd.read_html(str(team_stat))[0])
    for count, goaly_stat in enumerate(goaly_game_stats):
        goaly_df = pd.read_html(str(goaly_stat))[0]
        team_stats[count] = [team_stats[count], goaly_df]
    return team_stats


matches_dict = dict()

# quick ovveride:
premier_league_links = ['https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures']
for premiere_league_link in premier_league_links:
    premier_league_year = premier_league_link.split('/')[-2]
    fixtures = get_fixtures(premiere_league_link)
    match_report_links = [fbref_url.format(l.get('href')) for l in fixtures_table.find_all('a') if "Match Report" in l]
    print('LENGTH OF MATCH REPORT LINKS IS: '+str(len(match_report_links)))
    match_reports = list()
    
    for count, match_link  in enumerate(match_report_links):
        print(f"getting match data for match: {link}. Progress: {((count+1)/len(match_report_links))*100}%")
        match_data = get_match_data(match_link)
        match_reports.append(match_data)
        time.sleep(4)
        clear_output()
    matches_dict[premier_league_year] = match_reports

In [865]:
game_stats = matches_dict['9']
with open('game_stats.pkl', "wb") as f:
    pkl.dump(game_stats,f)

In [871]:
game_stats[189][0][0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
               Player                  #             Nation   
0     Dominic Solanke               19.0            eng ENG   
1         Timo Werner               16.0             de GER   
2       Son Heung-min                7.0             kr KOR   
3     Brennan Johnson               22.0            wls WAL   
4    Dejan Kulusevski               21.0             se SWE   
5     Pape Matar Sarr               29.0             sn SEN   
6       Yves Bissouma                8.0             ml MLI   
7      Lucas Bergvall               15.0             se SWE   
8      James Maddison               10.0            eng ENG   
9         Djed Spence               24.0            eng ENG   
10        Archie Gray               14.0            eng ENG   
11      Radu Drăgușin                6.0             ro ROU   
12    Sergio Reguilón                3.0             es ESP   
13        Pedro Porro               23.0             es ESP   
14     Brandon Austin               40.0            eng ENG   
15         15 Players                NaN                NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Performance      \
                  Pos                Age                Min         Gls Ast   
0                  FW             27-112                 90           1   0   
1                  LW             28-304                 61           0   0   
2                  LW             32-180                 29           0   0   
3                  RW             23-226                 90           0   0   
4               AM,RM             24-254                 90           0   0   
5                  DM             22-112                 61           0   0   
6                  CM             28-127                 29           0   0   
7                  DM             18-337                 61           0   0   
8                  LM             28-042                 29           0   0   
9               CB,LB             24-148                 90           0   0   
10                 CB             18-298                 90           0   0   
11                 CB             22-336                 45           0   0   
12                 LB             28-019                 45           0   0   
13                 RB             25-113                 90           0   1   
14                 GK             25-362                 90           0   0   
15                NaN                NaN                990           1   1   

             ... SCA     Passes                 Carries      Take-Ons       
   PK PKatt  ... SCA GCA    Cmp  Att  Cmp% PrgP Carries PrgC      Att Succ  
0   0     0  ...   2   0      9   17  52.9    1      15    1        2    0  
1   0     0  ...   0   0     11   16  68.8    3      11    3        1    0  
2   0     0  ...   2   0     24   27  88.9    2      24    2        4    1  
3   0     0  ...   5   1     17   22  77.3    3      19    3        2    1  
4   0     0  ...   5   0     28   38  73.7    7      38    5        1    1  
5   0     0  ...   2   0     27   33  81.8    4      23    0        1    0  
6   0     0  ...   0   0     29   30  96.7    4      22    2        0    0  
7   0     0  ...   0   0     29   36  80.6    3      27    3        3    1  
8   0     0  ...   1   0     31   35  88.6    7      30    2        0    0  
9   0     0  ...   0   0     46   61  75.4    4      40    0        2    0  
10  0     0  ...   0   0     46   53  86.8    2      36    0        0    0  
11  0     0  ...   0   0      9   16  56.3    0      13    0        0    0  
12  0     0  ...   0   0     30   41  73.2    4      28    0        2    1  
13  0     0  ...   5   1     39   55  70.9    2      33    2        0    0  
14  0     0  ...   0   0     22   32  68.8    0      22    0        0    0  
15  0     0  ...  22   2    397  512  77.5   46     381   23       18    5  

[16 rows x 31 columns]

In [875]:
x = get_fixtures('https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures')

In [877]:
pd.read_html(str(x))[0]

/var/folders/78/b2j3spl172n6y7p_sw2jggr40000gn/T/ipykernel_54417/126797153.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_html(str(x))[0]


,Wk,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes
0,1.0,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN
1,1.0,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN
2,1.0,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN
3,1.0,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth,29763.0,The City Ground,Michael Oliver,Match Report,NaN
4,1.0,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,38.0,Sun,2025-05-25,16:00,Fulham,NaN,NaN,NaN,Manchester City,NaN,Craven Cottage,NaN,Head-to-Head,NaN
414,38.0,Sun,2025-05-25,16:00,Nott'ham Forest,NaN,NaN,NaN,Chelsea,NaN,The City Ground,NaN,Head-to-Head,NaN
415,38.0,Sun,2025-05-25,16:00,Manchester Utd,NaN,NaN,NaN,Aston Villa,NaN,Old Trafford,NaN,Head-to-Head,NaN
416,38.0,Sun,2025-05-25,16:00,Wolves,NaN,NaN,NaN,Brentford,NaN,Molineux Stadium,NaN,Head-to-Head,NaN


In [804]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

NEO4J_URI='neo4j+s://2b521897.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='nCUihZktDGHjJUpQtKUal0IJhrk0eSuMdSFUm99Th3Q'
AURA_INSTANCEID='2b521897'
AURA_INSTANCENAME='Instance01'

In [803]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [677]:
# players_df = pd.read_csv('all_players_pd.csv')
print(players_data)




[{'Name': 'Virgil van Dijk', 'position': 'DF (CB, left)', 'footed': 'Right', 'height': '193cm', 'weight': '87kg', 'dob': '1991-07-08', 'birth_place': 'Breda, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 220,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e06683ca_2022.jpg'}, {'Name': 'Mohamed Salah', 'position': 'FW-MF (AM-WM, right)', 'footed': 'Left', 'height': '175cm', 'weight': '71kg', 'dob': '1992-06-15', 'birth_place': 'Basyūn, Egypt', 'national_team': 'Egypteg', 'club': 'Liverpool', 'wages': '￡ 350,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e342ad68_2022.jpg'}, {'Name': 'Ryan Gravenberch', 'position': 'MF (CM-DM-WM)', 'footed': 'Right', 'height': '190cm', 'weight': '78kg', 'dob': '2002-05-16', 'birth_place': 'Amsterdam, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 150,000 Weekly', 'contract_expiry

In [878]:
fixtures_2024 = get_fixtures('https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures')
fixtures_df = pd.read_html(str(fixtures_2024))[0]
fixtures_df = fixtures_df.rename(columns={'Wk':'GW'})
fixtures_df = fixtures_df[~fixtures_df['GW'].isnull()]
fixtures_df['GW'] = "GW"+fixtures_df['GW'].astype(int).astype(str)

/var/folders/78/b2j3spl172n6y7p_sw2jggr40000gn/T/ipykernel_54417/760373454.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  fixtures_df = pd.read_html(str(fixtures_2024))[0]


In [990]:
fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)

In [882]:
fixtures_df.to_csv('fixtures.csv')

In [839]:
match_reports[0][1][0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
               Player                  #             Nation   
0       Rodrigo Muniz                9.0             br BRA   
1        Raúl Jiménez                7.0             mx MEX   
2          Alex Iwobi               17.0             ng NGA   
3        Adama Traoré               11.0             es ESP   
4        Harry Wilson                8.0            wls WAL   
5    Emile Smith Rowe               32.0            eng ENG   
6         Tom Cairney               10.0            sct SCO   
7          Saša Lukić               20.0             rs SRB   
8      Jay Stansfield               28.0            eng ENG   
9     Andreas Pereira               18.0             br BRA   
10      Harrison Reed                6.0            eng ENG   
11   Antonee Robinson               33.0             us USA   
12      Calvin Bassey                3.0             ng NGA   
13          Issa Diop               31.0             fr FRA   
14         Kenny Tete                2.0             nl NED   
15         Bernd Leno                1.0             de GER   
16         16 Players                NaN                NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Performance      \
                  Pos                Age                Min         Gls Ast   
0                  FW             23-104                 77           0   0   
1                  FW             33-103                 13           0   0   
2                  LW             28-105                 90           0   0   
3                  RW             28-204                 77           0   0   
4                  RW             27-147                 13           0   0   
5                  AM             24-019                 63           0   0   
6               DM,AM             33-209                 27           0   0   
7                  DM             28-003                 89           0   0   
8               AM,DM             21-266                  1           0   0   
9               DM,AM             28-228                 89           0   0   
10                 DM             29-202                  1           0   0   
11                 LB             27-008                 90           0   0   
12                 CB             24-229                 90           0   0   
13                 CB             27-220                 90           0   0   
14                 RB             28-312                 90           0   0   
15                 GK             32-165                 90           0   0   
16                NaN                NaN                990           0   0   

             ... SCA     Passes                  Carries      Take-Ons       
   PK PKatt  ... SCA GCA    Cmp  Att   Cmp% PrgP Carries PrgC      Att Succ  
0   0     0  ...   2   0      5    6   83.3    1      11    0        1    1  
1   0     0  ...   0   0      2    4   50.0    0       2    1        0    0  
2   0     0  ...   0   0     19   29   65.5    2      31    1        5    1  
3   0     0  ...   4   0     14   21   66.7    0      25    5        9    4  
4   0     0  ...   0   0      5    6   83.3    0       6    1        0    0  
5   0     0  ...   0   0     20   27   74.1    1      23    0        0    0  
6   0     0  ...   1   0     15   17   88.2    4      14    2        3    2  
7   0     0  ...   0   0     40   45   88.9    4      30    1        0    0  
8   0     0  ...   0   0      0    0    NaN    0       0    0        0    0  
9   0     0  ...   9   0     37   55   67.3    5      28    1        2    0  
10  0     0  ...   0   0      1    1  100.0    0       1    0        0    0  
11  0     0  ...   0   0     32   42   76.2    2      31    2        2    0  
12  0     0  ...   1   0     57   62   91.9    5      51    1        0    0  
13  0     0  ...   0   0     28   35   80.0    2      26    0        0    0  
14  0     0  ...   2   0     24   32   75.0    2      2

In [735]:
# 
# (LEAGUE: PremLeague24-25) -> GWs -> Match -> Has_Home/Away
# 
# 


In [805]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLegue24-25"})

In [808]:
team_players = dict()
player_details = list()
for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        player_details.append(players[player])
    team_players[team] = current_team_players

In [809]:
with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img
    """
    
    session.run(query, players=player_details)

In [841]:
# clean up before sending
with open('match_reports.pkl','rb') as f:
    d = pkl.load(f)

In [886]:
len(match_reports)

199

In [887]:
fixtures_df

,GW,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes,Home_Score,Away_Score
0,GW1,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN,None,None
1,GW1,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN,None,None
2,GW1,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN,None,None
3,GW1,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth,29763.0,The City Ground,Michael Oliver,Match Report,NaN,None,None
4,GW1,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,GW38,Sun,2025-05-25,16:00,Fulham,NaN,NaN,NaN,Manchester City,NaN,Craven Cottage,NaN,Head-to-Head,NaN,None,None
414,GW38,Sun,2025-05-25,16:00,Nott'ham Forest,NaN,NaN,NaN,Chelsea,NaN,The City Ground,NaN,Head-to-Head,NaN,None,None
415,GW38,Sun,2025-05-25,16:00,Manchester Utd,NaN,NaN,NaN,Aston Villa,NaN,Old Trafford,NaN,Head-to-Head,NaN,None,None
416,GW38,Sun,2025-05-25,16:00,Wolves,NaN,NaN,NaN,Brentford,NaN,Molineux Stadium,NaN,Head-to-Head,NaN,None,None


In [902]:
fixtures_df.iloc[0]['Away']

'Fulham'

In [964]:
corrects = list()
for i in range(min(len(fixtures_df), len(match_reports))):

    home_team = fixtures_df.iloc[i]['Home']
    away_team = fixtures_df.iloc[i]['Away']

    home_team_player_team = all_players_dict[match_reports[i][0][0][('Unnamed: 0_level_0', 'Player')].iloc[0]]['club']
    away_team_player_team = all_players_dict[match_reports[i][1][0][('Unnamed: 0_level_0', 'Player')].iloc[0]]['club']

    if (home_team in home_team_player_team or home_team_player_team in home_team) and (away_team in away_team_player_team or away_team_player_team in away_team):
        continue
    else:
        
        print(fixtures_df.iloc[i])
        print(home_team)
        print(away_team)
        
        print(home_team_player_team)
        print(away_team_player_team)
        
        x = input('correct?')
        corrects.append(x)
        clear_output(wait=True)

In [927]:
players_team

{'Al Shabab (KSA)',
 'Anderlecht',
 'Arsenal',
 'Aston Villa',
 'Augsburg',
 'Auxerre',
 'Birmingham City',
 'Bournemouth',
 'Brentford',
 'Brighton & Hove Albion',
 'Burnley',
 'Chelsea',
 'Chesterfield',
 'Crystal Palace',
 'Derby County',
 'Everton',
 'Flamengo',
 'Fulham',
 'Hull City',
 'Ipswich Town',
 'Leicester City',
 'Liverpool',
 'Mallorca',
 'Manchester City',
 'Manchester United',
 'Marseille',
 'Middlesbrough',
 'Napoli',
 'Newcastle United',
 'Nottingham Forest',
 'Oxford United',
 'Portsmouth',
 'Queens Park Rangers',
 'Rennes',
 'Sevilla',
 'Southampton',
 'Stoke City',
 'Swansea City',
 'Tottenham Hotspur',
 'Werder Bremen',
 'West Bromwich Albion',
 'West Ham United',
 'Westerlo',
 'Wolverhampton Wanderers'}

In [965]:
# this is very important please do not delete
mappings = {"Brighton":'Brighton & Hove Albion',
            'Manchester Utd':'Manchester United',
            'Newcastle Utd':'Newcastle United',
            "Nott'ham Forest":'Nottingham Forest',
            'Tottenham':'Tottenham Hotspur',
            'West Ham':'West Ham United',
            'Wolves':'Wolverhampton Wanderers'}

In [935]:
for player, player_data in all_players_dict.items():
    club_name = player_data['club']  
    all_players_dict[player]['club'] = mappings[club_name] if club_name in mappings else club_name


In [960]:
fixtures_df = fixtures_df['Home'].map(mappings)
fixtures_df = fixtures_df['Away'].map(mappings)

In [951]:
fixtures_df = pd.read_csv('fixtures.csv')

In [984]:
fixtures_df.iloc[18]

Unnamed: 0                           19
GW                                  GW2
Day                                 Sun
Date                         2024-08-25
Time                              14:00
Home            Wolverhampton Wanderers
xG                                  1.9
Score                               2–6
xG.1                                1.6
Away                            Chelsea
Attendance                      31235.0
Venue                  Molineux Stadium
Referee                  Darren England
Match Report               Match Report
Notes                               NaN
Home_Score                          NaN
Away_Score                          NaN
Name: 18, dtype: object

In [1042]:
with driver.session(database="neo4j") as session:
    query = "match (n) detach delete n"
    session.run(query)

In [1043]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLeague"})

    query = "CREATE (s:Season {name: $name})"
    session.run(query, {"name":"24-25"})

    query = """
        MATCH (s:Season{name:$season_name}), (l:League{name: $league_name}) 
        CREATE (s)-[:SEASON_OF_LEAGUE]->(l)
        """
    print(session.run(query, {"league_name": "PremierLeague", "season_name": "24-25"}))

In [1044]:
team_players = dict()
player_details = list()

for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        players[player]['Team'] = team  # Fix the key access from player to players
        player_details.append(players[player])
    team_players[team] = team_players[team]

with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img

            MERGE (team:Team {name: player.Team})
            MERGE (p)-[:PLAYS_FOR]->(team)
    """
    
    session.run(query, players=player_details)


In [1028]:
player_details[0]

{'position': 'DF (CB, left)',
 'footed': 'Right',
 'height': '193cm',
 'weight': '87kg',
 'dob': '1991-07-08',
 'birth_place': 'Breda, Netherlands',
 'national_team': 'Netherlandsnl',
 'club': 'Liverpool',
 'wages': '￡ 220,000 Weekly',
 'contract_expiry': 'June 2025',
 'img': 'https://fbref.com/req/202302030/images/headshots/e06683ca_2022.jpg',
 'Name': 'Virgil van Dijk'}

In [1045]:
with driver.session(database="neo4j") as session:
    # Create GameWeek nodes and relationships
    for gw in range(1, 39):  # Loop from 1 to 38
        query = """
        MATCH (s:Season {name: $season_name})
        CREATE (gw:GameWeek {number: $number})-[:IS_A_GW]->(s)
        """
        session.run(query, {"season_name": "24-25", "number": gw})

In [1046]:
with driver.session(database="neo4j") as session:
    for _, row in fixtures_df.iterrows():
        query = """
        MERGE (match:Match {id: $match_id})
        SET match.date = $date,
            match.time = $time,
            match.home_team = $home_team,
            match.away_team = $away_team,
            match.home_expected = $home_expected,
            match.away_expected = $away_expected,
            match.home_score = $home_score,
            match.away_score = $away_score

        MERGE (ref:Referee {name: $referee})
        MERGE (venue:Venue {name: $venue})
        
        MERGE (home_team:Team {name: $home_team})
        MERGE (away_team:Team {name: $away_team})
        
        MERGE (match)-[:REFERRED_BY]->(ref)
        MERGE (match)-[:PLAYED_AT]->(venue)
        MERGE (match)-[:HOME_TEAM]->(home_team)
        MERGE (match)-[:AWAY_TEAM]->(away_team)
        """
        
        # Prepare the data dictionary
        data ={
            "match_id": f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}",
            "date": row["Date"],
            "time": row["Time"],
            "home_team": row["Home"],
            "away_team": row["Away"],
        }
        if row["Match Report"]:
            data["home_expected"]= row["xG"],
            data["away_expected"]= row["xG.1"],
            data["home_score"]= row["Home_Score"],
            data["away_score"]= row["Away_Score"],
            data["referee"]= row["Referee"],
            data["venue"] =  row["Venue"]
        
        # Execute the query
        session.run(query, data)


In [1015]:
teams = pd.concat([fixtures_df["Home"],fixtures_df["Away"]]).unique().tolist()
with driver.session(database="neo4j") as session:
    for team_name in teams:
        query = "CREATE (t:Team {name: $name})"
        session.run(query, {"name":team_name})

In [1047]:
with driver.session(database="neo4j") as session:
    for i, row in fixtures_df[:len(match_reports)].iterrows():
        match_id = f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}"
        home_team_name = row['Home']
        away_team_name = row['Away']

        # Add Home Team Player Stats
        for _, player_stats in home_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, nation: $nation, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[h:HOME_TEAM_PLAYER_STATS]->(player)
            SET h.goals = $goals,
                h.assists = $assists,
                h.penalties = $penalties,
                h.shots = $shots,
                h.shots_on_target = $shots_on_target,
                h.touches = $touches,
                h.tackles = $tackles,
                h.interceptions = $interceptions,
                h.blocks = $blocks,
                h.xg = $xg,
                h.npxg = $npxg,
                h.xag = $xag,
                h.sca = $sca,
                h.gca = $gca,
                h.passes_completed = $passes_completed,
                h.passes_attempted = $passes_attempted,
                h.pass_accuracy = $pass_accuracy,
                h.progressive_passes = $progressive_passes,
                h.carries = $carries,
                h.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": player_stats[('Unnamed: 2_level_0', 'Nation')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

        # Add Away Team Player Stats
        for _, player_stats in away_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, nation: $nation, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[a:AWAY_TEAM_PLAYER_STATS]->(player)
            SET a.goals = $goals,
                a.assists = $assists,
                a.penalties = $penalties,
                a.shots = $shots,
                a.shots_on_target = $shots_on_target,
                a.touches = $touches,
                a.tackles = $tackles,
                a.interceptions = $interceptions,
                a.blocks = $blocks,
                a.xg = $xg,
                a.npxg = $npxg,
                a.xag = $xag,
                a.sca = $sca,
                a.gca = $gca,
                a.passes_completed = $passes_completed,
                a.passes_attempted = $passes_attempted,
                a.pass_accuracy = $pass_accuracy,
                a.progressive_passes = $progressive_passes,
                a.carries = $carries,
                a.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": player_stats[('Unnamed: 2_level_0', 'Nation')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

         # Add Home Team Goalie Stats
        for _, goalie_stats in home_team_goaly_stats.iterrows():
            goalie_query = """
            MERGE (goalie:Player {name: $goalie_name, nation: $nation, position: "GK", age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (goalie)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[:HOME_TEAM_GOALIE_STATS]->(goalie)
            SET goalie.minutes_played = $minutes_played,
                goalie.sota = $sota,
                goalie.goals_allowed = $goals_allowed,
                goalie.saves = $saves,
                goalie.save_percentage = $save_percentage,
                goalie.psxg = $psxg,
                goalie.passes_completed = $passes_completed,
                goalie.passes_attempted = $passes_attempted,
                goalie.pass_accuracy = $pass_accuracy,
                goalie.gk_passes_attempted = $gk_passes_attempted,
                goalie.throws = $throws,
                goalie.launch_percentage = $launch_percentage,
                goalie.launch_average_length = $launch_average_length,
                goalie.goal_kicks_attempted = $goal_kicks_attempted,
                goalie.goal_kick_launch_percentage = $goal_kick_launch_percentage,
                goalie.goal_kick_average_length = $goal_kick_average_length,
                goalie.crosses_opportunities = $crosses_opportunities,
                goalie.crosses_stops = $crosses_stops,
                goalie.crosses_stop_percentage = $crosses_stop_percentage,
                goalie.opa = $opa,
                goalie.opa_average_distance = $opa_average_distance
            """
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": goalie_stats[('Unnamed: 1_level_0', 'Nation')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })

        # Add Away Team Goalie Stats
        for _, goalie_stats in away_team_goaly_stats.iterrows():
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": goalie_stats[('Unnamed: 1_level_0', 'Nation')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })


In [1072]:
query = """
MATCH (player:Player)-[:PLAYS_FOR]->(team:Team)
WITH player, COUNT(team) AS teamCount
WHERE teamCount > 3
RETURN player, teamCount
"""
players_errors = list()
with driver.session(database="neo4j") as session:
    r = session.run(query)
    for record in r:
        players_errors.append(record["player"]["name"])

In [1101]:
player_to_find = players_errors[0]
for i in range(len(match_reports)):
    if player_to_find in match_reports[i][0][0][("Unnamed: 0_level_0","Player")].tolist():
        print(match_reports[i][0][0])
        print('----------------------------------')
    if player_to_find in match_reports[i][1][0][("Unnamed: 0_level_0","Player")].tolist():
        print(match_reports[i][1][0])
        print('----------------------------------')

    Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                Player                  #             Nation   
0      Bruno Fernandes                8.0             pt POR   
1      Marcus Rashford               10.0            eng ENG   
2          Amad Diallo               16.0             ci CIV   
3   Alejandro Garnacho               17.0             ar ARG   
4          Mason Mount                7.0            eng ENG   
5       Joshua Zirkzee               11.0             nl NED   
6        Kobbie Mainoo               37.0            eng ENG   
7      Scott McTominay               39.0            sct SCO   
8             Casemiro               18.0             br BRA   
9          Diogo Dalot               20.0             pt POR   
10   Lisandro Martínez                6.0             ar ARG   
11       Harry Maguire                5.0            eng ENG   
12         Jonny Evans               35.0            nir NIR   
13   Noussair Mazraoui                3.

In [1104]:
countries = list()
for p, v in all_players_dict.items():
    countries.append(v['national_team'])

In [1107]:
with open("match_reports.pkl","wb") as f:
    pkl.dump(match_reports,f)

In [1106]:
len(countries)

91